In [1]:
import pandas as pd
import pickle

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from imblearn.over_sampling import SMOTE

In [2]:
# Load dataset
df = pd.read_csv("../notebook/cleaned_data.csv")

In [3]:
# Features and target
X = df.drop("is_fraud", axis=1)

y = df["is_fraud"]

In [4]:
# Split FIRST
X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [5]:
# Column groups
categorical_features = ["merchant_category"]

numerical_features = [

    "transaction_id",
    "amount",
    "transaction_hour",
    "foreign_transaction",
    "location_mismatch",
    "device_trust_score",
    "velocity_last_24h",
    "cardholder_age"
]

In [6]:
# Preprocessor
preprocessor = ColumnTransformer(

    transformers=[

        (
            "num",
            StandardScaler(),
            numerical_features
        ),

        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

In [7]:
# Fit ONLY on train data
X_train_processed = preprocessor.fit_transform(X_train)

# Transform test separately
X_test_processed = preprocessor.transform(X_test)

In [8]:
# SMOTE ONLY on train data
smote = SMOTE(random_state=42)

X_train_resampled, y_train_resampled = smote.fit_resample(

    X_train_processed,
    y_train
)

In [9]:
# Train model
model = RandomForestClassifier(

    n_estimators=100,
    max_depth=10,
    random_state=42
)

model.fit(X_train_resampled, y_train_resampled)

# Predict on untouched test set
y_pred = model.predict(X_test_processed)


In [10]:
import pandas as pd

# Convert training data to DataFrame
train_df = pd.DataFrame(X_train_resampled)

# Add target column
train_df['target'] = y_train_resampled

# Save training data
train_df.to_csv('../notebook/train.csv', index=False)

print("train.csv saved successfully")


# Convert test data to DataFrame
test_df = pd.DataFrame(X_test_processed)

# Add actual target values
test_df['target'] = y_test

# Add predicted values
test_df['prediction'] = y_pred

# Save test data
test_df.to_csv('../notebook/test.csv', index=False)


print("test.csv saved successfully")

train.csv saved successfully
test.csv saved successfully


In [11]:
# Metrics
print("Accuracy :", accuracy_score(y_test, y_pred))

print("Precision :", precision_score(y_test, y_pred))

print("Recall :", recall_score(y_test, y_pred))

print("F1 :", f1_score(y_test, y_pred))


Accuracy : 0.995
Precision : 0.9166666666666666
Recall : 0.7333333333333333
F1 : 0.8148148148148148


In [12]:
# Save files
pickle.dump(model, open("../model.pkl", "wb"))

pickle.dump(preprocessor, open("../preprocessor.pkl", "wb"))